In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/unified_occupancy_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (90144, 13)


,source,recording_id,timestamp,time_of_day,room,room_type,room_capacity,occupant_count,occupancy_ratio,co2,temperature,humidity,illuminance
0,ASHRAE,ASHRAE_Day_0,NaN,0 days 00:00:00,1,Lecture room,84,0.0,0.0,NaN,23.6,52.941177,0.0
1,ASHRAE,ASHRAE_Day_0,NaN,0 days 00:05:00,1,Lecture room,84,0.0,0.0,477.760010,23.6,52.941177,0.0
2,ASHRAE,ASHRAE_Day_0,NaN,0 days 00:10:00,1,Lecture room,84,0.0,0.0,480.959991,23.6,NaN,NaN
3,ASHRAE,ASHRAE_Day_0,NaN,0 days 00:15:00,1,Lecture room,84,0.0,0.0,478.720001,NaN,NaN,NaN
4,ASHRAE,ASHRAE_Day_0,NaN,0 days 00:20:00,1,Lecture room,84,0.0,0.0,475.839996,NaN,NaN,NaN


In [2]:
# Convert time_of_day into a timedelta
df["time_of_day"] = pd.to_timedelta(df["time_of_day"])

# Sort by source, recording session, room, and time
df = df.sort_values(
    ["source", "recording_id", "room", "time_of_day"]
).reset_index(drop=True)

# Check recording session sizes
session_sizes = (
    df.groupby(["source", "recording_id", "room"])
    .size()
)

print("Number of recording sessions:", len(session_sizes))
print("\nSession size distribution:")
display(session_sizes.value_counts().sort_index())

print("\nMinimum intervals per session:", session_sizes.min())
print("Maximum intervals per session:", session_sizes.max())

Number of recording sessions: 313

Session size distribution:


288    313
Name: count, dtype: int64


Minimum intervals per session: 288
Maximum intervals per session: 288


In [3]:
# Predict occupancy ratio 30 minutes into the future
df["target_occupancy_ratio_30min"] = (
    df.groupby(["source", "recording_id", "room"])["occupancy_ratio"]
    .shift(-6)
)

# Check target creation
print("Total rows:", len(df))

print(
    "Rows with target:",
    df["target_occupancy_ratio_30min"].notna().sum()
)

print(
    "Rows without target:",
    df["target_occupancy_ratio_30min"].isna().sum()
)

# Inspect one recording session
display(
    df[df["recording_id"] == "ASHRAE_Day_0"][
        ["time_of_day", "occupancy_ratio", "target_occupancy_ratio_30min"]
    ].tail(10)
)

Total rows: 90144
Rows with target: 88266
Rows without target: 1878


,time_of_day,occupancy_ratio,target_occupancy_ratio_30min
854,0 days 23:10:00,0.0,0.0
855,0 days 23:15:00,0.0,0.0
856,0 days 23:20:00,0.0,0.0
857,0 days 23:25:00,0.0,0.0
858,0 days 23:30:00,0.0,NaN
859,0 days 23:35:00,0.0,NaN
860,0 days 23:40:00,0.0,NaN
861,0 days 23:45:00,0.0,NaN
862,0 days 23:50:00,0.0,NaN
863,0 days 23:55:00,0.0,NaN


In [4]:
# Keep only rows with a valid 30-minute target
model_df = df.dropna(
    subset=["target_occupancy_ratio_30min"]
).copy()

# Reset index
model_df = model_df.reset_index(drop=True)

print("Modeling dataset shape:", model_df.shape)

print("\nMissing target values:")
print(model_df["target_occupancy_ratio_30min"].isna().sum())

print("\nTarget distribution:")
display(
    model_df["target_occupancy_ratio_30min"]
    .describe()
    .round(3)
)

Modeling dataset shape: (88266, 14)

Missing target values:
0

Target distribution:


count    88266.000
mean         0.094
std          0.169
min          0.000
25%          0.000
50%          0.000
75%          0.120
max          2.375
Name: target_occupancy_ratio_30min, dtype: float64

In [5]:
# Correlation between current and future occupancy
correlation = model_df[
    ["occupancy_ratio", "target_occupancy_ratio_30min"]
].corr()

print("Current vs. future occupancy correlation:")
display(correlation.round(3))

# Compare current and future occupancy statistics
comparison = model_df[
    ["occupancy_ratio", "target_occupancy_ratio_30min"]
].describe().round(3)

display(comparison)

Current vs. future occupancy correlation:


,occupancy_ratio,target_occupancy_ratio_30min
occupancy_ratio,1.000,0.893
target_occupancy_ratio_30min,0.893,1.000


,occupancy_ratio,target_occupancy_ratio_30min
count,88266.000,88266.000
mean,0.094,0.094
std,0.169,0.169
min,0.000,0.000
25%,0.000,0.000
50%,0.000,0.000
75%,0.120,0.120
max,2.375,2.375


In [6]:
# Define recording session groups
groups = ["source", "recording_id", "room"]

# Create historical occupancy features
for minutes in [5, 15, 30, 60]:
    periods = minutes // 5

    model_df[f"occupancy_lag_{minutes}min"] = (
        model_df.groupby(groups)["occupancy_ratio"]
        .shift(periods)
    )

# Inspect the new features
display(
    model_df[
        [
            "source",
            "recording_id",
            "room",
            "time_of_day",
            "occupancy_ratio",
            "occupancy_lag_5min",
            "occupancy_lag_15min",
            "occupancy_lag_30min",
            "occupancy_lag_60min",
            "target_occupancy_ratio_30min"
        ]
    ].head(15)
)

,source,recording_id,room,time_of_day,occupancy_ratio,occupancy_lag_5min,occupancy_lag_15min,occupancy_lag_30min,occupancy_lag_60min,target_occupancy_ratio_30min
0,ASHRAE,ASHRAE_Day_0,1,0 days 00:00:00,0.0,NaN,NaN,NaN,NaN,0.0
1,ASHRAE,ASHRAE_Day_0,1,0 days 00:05:00,0.0,0.0,NaN,NaN,NaN,0.0
2,ASHRAE,ASHRAE_Day_0,1,0 days 00:10:00,0.0,0.0,NaN,NaN,NaN,0.0
3,ASHRAE,ASHRAE_Day_0,1,0 days 00:15:00,0.0,0.0,0.0,NaN,NaN,0.0
4,ASHRAE,ASHRAE_Day_0,1,0 days 00:20:00,0.0,0.0,0.0,NaN,NaN,0.0
5,ASHRAE,ASHRAE_Day_0,1,0 days 00:25:00,0.0,0.0,0.0,NaN,NaN,0.0
6,ASHRAE,ASHRAE_Day_0,1,0 days 00:30:00,0.0,0.0,0.0,0.0,NaN,0.0
7,ASHRAE,ASHRAE_Day_0,1,0 days 00:35:00,0.0,0.0,0.0,0.0,NaN,0.0
8,ASHRAE,ASHRAE_Day_0,1,0 days 00:40:00,0.0,0.0,0.0,0.0,NaN,0.0
9,ASHRAE,ASHRAE_Day_0,1,0 days 00:45:00,0.0,0.0,0.0,0.0,NaN,0.0


In [7]:
# Create historical CO2 features
for minutes in [5, 15, 30, 60]:
    periods = minutes // 5

    model_df[f"co2_lag_{minutes}min"] = (
        model_df.groupby(groups)["co2"]
        .shift(periods)
    )

# Inspect the new features
display(
    model_df[
        [
            "time_of_day",
            "co2",
            "co2_lag_5min",
            "co2_lag_15min",
            "co2_lag_30min",
            "co2_lag_60min"
        ]
    ].head(15)
)

,time_of_day,co2,co2_lag_5min,co2_lag_15min,co2_lag_30min,co2_lag_60min
0,0 days 00:00:00,NaN,NaN,NaN,NaN,NaN
1,0 days 00:05:00,477.760010,NaN,NaN,NaN,NaN
2,0 days 00:10:00,480.959991,477.760010,NaN,NaN,NaN
3,0 days 00:15:00,478.720001,480.959991,NaN,NaN,NaN
4,0 days 00:20:00,475.839996,478.720001,477.760010,NaN,NaN
5,0 days 00:25:00,483.839996,475.839996,480.959991,NaN,NaN
6,0 days 00:30:00,476.799988,483.839996,478.720001,NaN,NaN
7,0 days 00:35:00,482.880005,476.799988,475.839996,477.760010,NaN
8,0 days 00:40:00,480.000000,482.880005,483.839996,480.959991,NaN
9,0 days 00:45:00,475.839996,480.000000,476.799988,478.720001,NaN


In [8]:
# Create historical temperature features
for minutes in [5, 15, 30, 60]:
    periods = minutes // 5

    model_df[f"temperature_lag_{minutes}min"] = (
        model_df.groupby(groups)["temperature"]
        .shift(periods)
    )

# Inspect the new features
display(
    model_df[
        [
            "time_of_day",
            "temperature",
            "temperature_lag_5min",
            "temperature_lag_15min",
            "temperature_lag_30min",
            "temperature_lag_60min"
        ]
    ].head(15)
)

,time_of_day,temperature,temperature_lag_5min,temperature_lag_15min,temperature_lag_30min,temperature_lag_60min
0,0 days 00:00:00,23.6,NaN,NaN,NaN,NaN
1,0 days 00:05:00,23.6,23.6,NaN,NaN,NaN
2,0 days 00:10:00,23.6,23.6,NaN,NaN,NaN
3,0 days 00:15:00,NaN,23.6,23.6,NaN,NaN
4,0 days 00:20:00,NaN,NaN,23.6,NaN,NaN
5,0 days 00:25:00,NaN,NaN,23.6,NaN,NaN
6,0 days 00:30:00,NaN,NaN,NaN,23.6,NaN
7,0 days 00:35:00,NaN,NaN,NaN,23.6,NaN
8,0 days 00:40:00,NaN,NaN,NaN,23.6,NaN
9,0 days 00:45:00,NaN,NaN,NaN,NaN,NaN


In [9]:
# Create historical humidity features
for minutes in [5, 15, 30, 60]:
    periods = minutes // 5

    model_df[f"humidity_lag_{minutes}min"] = (
        model_df.groupby(groups)["humidity"]
        .shift(periods)
    )

# Inspect the new features
display(
    model_df[
        [
            "time_of_day",
            "humidity",
            "humidity_lag_5min",
            "humidity_lag_15min",
            "humidity_lag_30min",
            "humidity_lag_60min"
        ]
    ].head(15)
)

,time_of_day,humidity,humidity_lag_5min,humidity_lag_15min,humidity_lag_30min,humidity_lag_60min
0,0 days 00:00:00,52.941177,NaN,NaN,NaN,NaN
1,0 days 00:05:00,52.941177,52.941177,NaN,NaN,NaN
2,0 days 00:10:00,NaN,52.941177,NaN,NaN,NaN
3,0 days 00:15:00,NaN,NaN,52.941177,NaN,NaN
4,0 days 00:20:00,NaN,NaN,52.941177,NaN,NaN
5,0 days 00:25:00,NaN,NaN,NaN,NaN,NaN
6,0 days 00:30:00,NaN,NaN,NaN,52.941177,NaN
7,0 days 00:35:00,NaN,NaN,NaN,52.941177,NaN
8,0 days 00:40:00,NaN,NaN,NaN,NaN,NaN
9,0 days 00:45:00,NaN,NaN,NaN,NaN,NaN


In [10]:
# Create historical illuminance features
for minutes in [5, 15, 30, 60]:
    periods = minutes // 5

    model_df[f"illuminance_lag_{minutes}min"] = (
        model_df.groupby(groups)["illuminance"]
        .shift(periods)
    )

# Inspect the new features
display(
    model_df[
        [
            "time_of_day",
            "illuminance",
            "illuminance_lag_5min",
            "illuminance_lag_15min",
            "illuminance_lag_30min",
            "illuminance_lag_60min"
        ]
    ].head(15)
)

,time_of_day,illuminance,illuminance_lag_5min,illuminance_lag_15min,illuminance_lag_30min,illuminance_lag_60min
0,0 days 00:00:00,0.0,NaN,NaN,NaN,NaN
1,0 days 00:05:00,0.0,0.0,NaN,NaN,NaN
2,0 days 00:10:00,NaN,0.0,NaN,NaN,NaN
3,0 days 00:15:00,NaN,NaN,0.0,NaN,NaN
4,0 days 00:20:00,NaN,NaN,0.0,NaN,NaN
5,0 days 00:25:00,NaN,NaN,NaN,NaN,NaN
6,0 days 00:30:00,NaN,NaN,NaN,0.0,NaN
7,0 days 00:35:00,NaN,NaN,NaN,0.0,NaN
8,0 days 00:40:00,NaN,NaN,NaN,NaN,NaN
9,0 days 00:45:00,NaN,NaN,NaN,NaN,NaN


In [11]:
# Calculate missing values and percentages
missing_summary = pd.DataFrame({
    "missing_count": model_df.isna().sum(),
    "missing_percentage": (
        model_df.isna().mean() * 100
    ).round(2)
})

# Sort by missing percentage
missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percentage
timestamp,37224,42.17
humidity_lag_60min,33449,37.90
humidity_lag_30min,32319,36.62
humidity_lag_15min,31751,35.97
humidity_lag_5min,31369,35.54
humidity,31183,35.33
temperature_lag_60min,24248,27.47
temperature_lag_30min,22954,26.01
temperature_lag_15min,22311,25.28
temperature_lag_5min,21899,24.81


In [12]:
# Compare missing percentages by source
sensor_columns = [
    "co2",
    "co2_lag_5min",
    "co2_lag_15min",
    "co2_lag_30min",
    "co2_lag_60min",
    "temperature",
    "humidity",
    "illuminance",
    "occupancy_lag_5min",
    "occupancy_lag_15min",
    "occupancy_lag_30min",
    "occupancy_lag_60min"
]

missing_by_source = (
    model_df.groupby("source")[sensor_columns]
    .apply(lambda group: group.isna().mean() * 100)
    .round(2)
)

display(missing_by_source.T)

source,ASHRAE,ROBOD
co2,6.39,0.17
co2_lag_5min,6.72,0.52
co2_lag_15min,7.38,1.23
co2_lag_30min,8.37,2.29
co2_lag_60min,10.31,4.42
temperature,58.25,0.00
humidity,83.54,0.17
illuminance,50.33,0.17
occupancy_lag_5min,0.35,0.35
occupancy_lag_15min,1.06,1.06


In [13]:
# Convert time of day into decimal hours
model_df["hour"] = (
    model_df["time_of_day"].dt.total_seconds() / 3600
)

# Create cyclical time features
model_df["hour_sin"] = np.sin(
    2 * np.pi * model_df["hour"] / 24
)

model_df["hour_cos"] = np.cos(
    2 * np.pi * model_df["hour"] / 24
)

# Inspect the results
display(
    model_df[
        ["time_of_day", "hour", "hour_sin", "hour_cos"]
    ].head(15)
)

,time_of_day,hour,hour_sin,hour_cos
0,0 days 00:00:00,0.000000,0.000000,1.000000
1,0 days 00:05:00,0.083333,0.021815,0.999762
2,0 days 00:10:00,0.166667,0.043619,0.999048
3,0 days 00:15:00,0.250000,0.065403,0.997859
4,0 days 00:20:00,0.333333,0.087156,0.996195
5,0 days 00:25:00,0.416667,0.108867,0.994056
6,0 days 00:30:00,0.500000,0.130526,0.991445
7,0 days 00:35:00,0.583333,0.152123,0.988362
8,0 days 00:40:00,0.666667,0.173648,0.984808
9,0 days 00:45:00,0.750000,0.195090,0.980785


In [14]:
# Check room information
room_summary = (
    model_df.groupby(["source", "room", "room_type", "room_capacity"])
    .size()
    .reset_index(name="row_count")
)

display(room_summary)

,source,room,room_type,room_capacity,row_count
0,ASHRAE,1,Lecture room,84,12408
1,ASHRAE,2,Study zone,32,12408
2,ASHRAE,3,Study zone,32,12408
3,ROBOD,1,Lecture room,40,8178
4,ROBOD,2,Lecture room,40,8178
5,ROBOD,3,Office,15,8178
6,ROBOD,4,Office,25,13254
7,ROBOD,5,Library,36,13254


In [15]:
# Create a group identifier for train/test splitting
model_df["split_group"] = (
    model_df["source"] + "_" + model_df["recording_id"]
)

# Check the number of unique recording groups
print(
    "Unique recording groups:",
    model_df["split_group"].nunique()
)

# Check groups by source
print("\nRecording groups by source:")
display(
    model_df.groupby("source")["split_group"].nunique()
)

# Check rows per group
group_sizes = model_df.groupby("split_group").size()

print("\nRecording group size summary:")
display(group_sizes.describe())

Unique recording groups: 91

Recording groups by source:


source
ASHRAE    44
ROBOD     47
Name: split_group, dtype: int64


Recording group size summary:


count      91.000000
mean      969.956044
std       320.728196
min       564.000000
25%       846.000000
50%       846.000000
75%      1410.000000
max      1410.000000
dtype: float64

In [16]:
from sklearn.model_selection import GroupShuffleSplit

# Define model features
feature_columns = [
    "occupancy_ratio",
    "occupancy_lag_5min",
    "occupancy_lag_15min",
    "occupancy_lag_30min",
    "occupancy_lag_60min",
    "co2",
    "co2_lag_5min",
    "co2_lag_15min",
    "co2_lag_30min",
    "co2_lag_60min",
    "hour_sin",
    "hour_cos",
    "room_capacity",
    "room_type"
]

target_column = "target_occupancy_ratio_30min"

# Prepare features and target
X = model_df[feature_columns].copy()
y = model_df[target_column].copy()
groups = model_df["split_group"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Missing feature values:", X.isna().sum().sum())
print("Missing target values:", y.isna().sum())

Feature shape: (88266, 14)
Target shape: (88266,)
Missing feature values: 25874
Missing target values: 0


In [17]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining groups:", train_groups.nunique())
print("Testing groups:", test_groups.nunique())

print("\nTraining source distribution:")
display(model_df.iloc[train_idx]["source"].value_counts())

print("\nTesting source distribution:")
display(model_df.iloc[test_idx]["source"].value_counts())

print(
    "\nShared groups:",
    len(set(train_groups.unique()) & set(test_groups.unique()))
)

Training rows: 70218
Testing rows: 18048

Training groups: 72
Testing groups: 19

Training source distribution:


source
ROBOD     42300
ASHRAE    27918
Name: count, dtype: int64


Testing source distribution:


source
ASHRAE    9306
ROBOD     8742
Name: count, dtype: int64


Shared groups: 0


In [18]:
# Count unique recording groups per source in each split
train_group_summary = (
    model_df.iloc[train_idx]
    .groupby("source")["split_group"]
    .nunique()
)

test_group_summary = (
    model_df.iloc[test_idx]
    .groupby("source")["split_group"]
    .nunique()
)

group_distribution = pd.DataFrame({
    "training_groups": train_group_summary,
    "testing_groups": test_group_summary
})

display(group_distribution)

,training_groups,testing_groups
source,,
ASHRAE,33,11
ROBOD,39,8


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Separate numerical and categorical features
numeric_features = [
    col for col in feature_columns
    if col != "room_type"
]

categorical_features = ["room_type"]

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 13
Categorical features: 1


In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import time

# Build the complete model pipeline
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
start_time = time.time()

rf_model.fit(X_train, y_train)

training_time = time.time() - start_time

# Predict on unseen test data
y_pred = rf_model.predict(X_test)

# Evaluate performance
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Training time: {training_time:.2f} seconds")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

Training time: 16.70 seconds
MAE:  0.0333
RMSE: 0.0755
R²:   0.7827


In [21]:
# Create a results table
evaluation_df = model_df.iloc[test_idx][
    ["source", "room", "room_type"]
].copy()

evaluation_df["actual"] = y_test.values
evaluation_df["predicted"] = y_pred

# Calculate metrics separately for each source
source_results = []

for source, group in evaluation_df.groupby("source"):

    mae = mean_absolute_error(
        group["actual"], group["predicted"]
    )

    rmse = np.sqrt(mean_squared_error(
        group["actual"], group["predicted"]
    ))

    r2 = r2_score(
        group["actual"], group["predicted"]
    )

    source_results.append({
        "source": source,
        "test_rows": len(group),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

source_metrics = pd.DataFrame(source_results)

display(source_metrics.round(4))

,source,test_rows,MAE,RMSE,R2
0,ASHRAE,9306,0.0375,0.0783,0.8080
1,ROBOD,8742,0.0287,0.0725,0.7366


In [22]:
room_results = []

for (source, room, room_type), group in evaluation_df.groupby(
    ["source", "room", "room_type"]
):

    mae = mean_absolute_error(
        group["actual"], group["predicted"]
    )

    rmse = np.sqrt(mean_squared_error(
        group["actual"], group["predicted"]
    ))

    r2 = r2_score(
        group["actual"], group["predicted"]
    )

    room_results.append({
        "source": source,
        "room": room,
        "room_type": room_type,
        "test_rows": len(group),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

room_metrics = pd.DataFrame(room_results)

display(
    room_metrics.sort_values(
        ["source", "room"]
    ).round(4)
)

,source,room,room_type,test_rows,MAE,RMSE,R2
0,ASHRAE,1,Lecture room,3102,0.0320,0.0729,0.7638
1,ASHRAE,2,Study zone,3102,0.0417,0.0843,0.7911
2,ASHRAE,3,Study zone,3102,0.0389,0.0771,0.8414
3,ROBOD,1,Lecture room,1410,0.0401,0.1272,0.6302
4,ROBOD,2,Lecture room,1410,0.0299,0.0803,0.5378
5,ROBOD,3,Office,1410,0.0328,0.0639,0.7826
6,ROBOD,4,Office,2256,0.0319,0.0517,0.8538
7,ROBOD,5,Library,2256,0.0152,0.0316,0.8538


In [23]:
# Persistence baseline:
# Predict future occupancy using current occupancy
baseline_pred = X_test["occupancy_ratio"].values

# Overall baseline metrics
baseline_mae = mean_absolute_error(y_test, baseline_pred)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)

baseline_r2 = r2_score(y_test, baseline_pred)

print("Persistence Baseline")
print(f"MAE:  {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R2:   {baseline_r2:.4f}")

print("\nRandom Forest")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"R2:   {r2_score(y_test, y_pred):.4f}")

Persistence Baseline
MAE:  0.0300
RMSE: 0.0791
R2:   0.7616

Random Forest
MAE:  0.0333
RMSE: 0.0755
R2:   0.7827


In [24]:
baseline_results = evaluation_df.copy()

baseline_results["baseline_predicted"] = baseline_pred

source_comparison = []

for source, group in baseline_results.groupby("source"):

    rf_mae = mean_absolute_error(
        group["actual"], group["predicted"]
    )

    baseline_mae = mean_absolute_error(
        group["actual"], group["baseline_predicted"]
    )

    rf_rmse = np.sqrt(mean_squared_error(
        group["actual"], group["predicted"]
    ))

    baseline_rmse = np.sqrt(mean_squared_error(
        group["actual"], group["baseline_predicted"]
    ))

    source_comparison.append({
        "source": source,
        "RF_MAE": rf_mae,
        "Baseline_MAE": baseline_mae,
        "RF_RMSE": rf_rmse,
        "Baseline_RMSE": baseline_rmse
    })

display(pd.DataFrame(source_comparison).round(4))

,source,RF_MAE,Baseline_MAE,RF_RMSE,Baseline_RMSE
0,ASHRAE,0.0375,0.0341,0.0783,0.0834
1,ROBOD,0.0287,0.0256,0.0725,0.0742


In [25]:
room_comparison = []

for (source, room, room_type), group in baseline_results.groupby(
    ["source", "room", "room_type"]
):

    rf_mae = mean_absolute_error(
        group["actual"], group["predicted"]
    )

    baseline_mae = mean_absolute_error(
        group["actual"], group["baseline_predicted"]
    )

    rf_rmse = np.sqrt(mean_squared_error(
        group["actual"], group["predicted"]
    ))

    baseline_rmse = np.sqrt(mean_squared_error(
        group["actual"], group["baseline_predicted"]
    ))

    room_comparison.append({
        "source": source,
        "room": room,
        "room_type": room_type,
        "RF_MAE": rf_mae,
        "Baseline_MAE": baseline_mae,
        "RF_RMSE": rf_rmse,
        "Baseline_RMSE": baseline_rmse
    })

room_comparison_df = pd.DataFrame(room_comparison)

display(room_comparison_df.round(4))

,source,room,room_type,RF_MAE,Baseline_MAE,RF_RMSE,Baseline_RMSE
0,ASHRAE,1,Lecture room,0.0320,0.0269,0.0729,0.0809
1,ASHRAE,2,Study zone,0.0417,0.0394,0.0843,0.0897
2,ASHRAE,3,Study zone,0.0389,0.0360,0.0771,0.0793
3,ROBOD,1,Lecture room,0.0401,0.0255,0.1272,0.1105
4,ROBOD,2,Lecture room,0.0299,0.0315,0.0803,0.1020
5,ROBOD,3,Office,0.0328,0.0287,0.0639,0.0693
6,ROBOD,4,Office,0.0319,0.0316,0.0517,0.0561
7,ROBOD,5,Library,0.0152,0.0141,0.0316,0.0323


In [26]:
# Select occupancy and time features only
occupancy_features = [
    "occupancy_ratio",
    "occupancy_lag_5min",
    "occupancy_lag_15min",
    "occupancy_lag_30min",
    "occupancy_lag_60min",
    "hour_sin",
    "hour_cos",
    "room_capacity",
    "room_type"
]

# Separate numerical and categorical features
occupancy_numeric = [
    col for col in occupancy_features
    if col != "room_type"
]

occupancy_categorical = ["room_type"]

# Create preprocessing
occupancy_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), occupancy_numeric),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), occupancy_categorical)
    ]
)

# Prepare training and testing data
X_train_occ = X_train[occupancy_features]
X_test_occ = X_test[occupancy_features]

# Build model
occupancy_rf = Pipeline(steps=[
    ("preprocessor", occupancy_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
occupancy_rf.fit(X_train_occ, y_train)

# Predict
y_pred_occ = occupancy_rf.predict(X_test_occ)

# Evaluate
print("Occupancy-only Random Forest")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_occ):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_occ)):.4f}")
print(f"R2:   {r2_score(y_test, y_pred_occ):.4f}")

Occupancy-only Random Forest
MAE:  0.0326
RMSE: 0.0766
R2:   0.7766


In [27]:
# Get the trained Random Forest
rf_regressor = rf_model.named_steps["regressor"]

# Get transformed feature names
transformed_feature_names = (
    rf_model.named_steps["preprocessor"].get_feature_names_out()
)

# Extract feature importance
importance_df = pd.DataFrame({
    "feature": transformed_feature_names,
    "importance": rf_regressor.feature_importances_
})

# Sort from highest to lowest
importance_df = importance_df.sort_values(
    "importance", ascending=False
)

display(importance_df.round(4))

,feature,importance
0,num__occupancy_ratio,0.8109
10,num__hour_sin,0.0324
9,num__co2_lag_60min,0.0167
11,num__hour_cos,0.0161
3,num__occupancy_lag_30min,0.0155
2,num__occupancy_lag_15min,0.0154
8,num__co2_lag_30min,0.0150
4,num__occupancy_lag_60min,0.0146
1,num__occupancy_lag_5min,0.0144
5,num__co2,0.0129


In [28]:
# Remove current occupancy from the feature list
no_current_features = [
    col for col in feature_columns
    if col != "occupancy_ratio"
]

# Prepare training and testing data
X_train_no_current = X_train[no_current_features]
X_test_no_current = X_test[no_current_features]

# Separate numerical and categorical features
no_current_numeric = [
    col for col in no_current_features
    if col != "room_type"
]

no_current_categorical = ["room_type"]

# Preprocessing
no_current_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), no_current_numeric),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), no_current_categorical)
    ]
)

# Model
no_current_rf = Pipeline(steps=[
    ("preprocessor", no_current_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# Train and predict
no_current_rf.fit(X_train_no_current, y_train)
y_pred_no_current = no_current_rf.predict(X_test_no_current)

# Evaluate
print("Random Forest without current occupancy")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_no_current):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_no_current)):.4f}")
print(f"R2:   {r2_score(y_test, y_pred_no_current):.4f}")

Random Forest without current occupancy
MAE:  0.0355
RMSE: 0.0795
R2:   0.7593


In [30]:
change_df = baseline_results.copy()

# Calculate actual occupancy change over 30 minutes
change_df["occupancy_change"] = (
    change_df["actual"] -
    model_df.iloc[test_idx]["occupancy_ratio"].values
)


# Categorize the change
change_df["change_type"] = np.select(
    [
        change_df["occupancy_change"] > 0.05,
        change_df["occupancy_change"] < -0.05
    ],
    [
        "Increase",
        "Decrease"
    ],
    default="Stable"
)

# Evaluate each category
change_results = []

for change_type, group in change_df.groupby("change_type"):

    change_results.append({
        "change_type": change_type,
        "rows": len(group),
        "RF_MAE": mean_absolute_error(
            group["actual"], group["predicted"]
        ),
        "Baseline_MAE": mean_absolute_error(
            group["actual"], group["baseline_predicted"]
        ),
        "RF_RMSE": np.sqrt(mean_squared_error(
            group["actual"], group["predicted"]
        )),
        "Baseline_RMSE": np.sqrt(mean_squared_error(
            group["actual"], group["baseline_predicted"]
        ))
    })

display(pd.DataFrame(change_results).round(4))

,change_type,rows,RF_MAE,Baseline_MAE,RF_RMSE,Baseline_RMSE
0,Decrease,1574,0.1119,0.1481,0.1535,0.1864
1,Increase,1569,0.1122,0.1475,0.1581,0.1884
2,Stable,14905,0.0167,0.0051,0.0423,0.0130


In [32]:
# Recreate the feature sets using model_df
X_train_env = model_df.iloc[train_idx][environment_features].copy()
X_test_env = model_df.iloc[test_idx][environment_features].copy()

print("Training shape:", X_train_env.shape)
print("Testing shape:", X_test_env.shape)

Training shape: (70218, 24)
Testing shape: (18048, 24)


In [38]:



environment_rf.fit(X_train_env, y_train)

y_pred_env = environment_rf.predict(X_test_env)

print("RF with CO2, temperature and humidity")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_env):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_env)):.4f}")
print(f"R2:   {r2_score(y_test, y_pred_env):.4f}")

RF with CO2, temperature and humidity
MAE:  0.0340
RMSE: 0.0767
R2:   0.7759


In [39]:
illuminance_features = [
    "illuminance",
    "illuminance_lag_5min",
    "illuminance_lag_15min",
    "illuminance_lag_30min",
    "illuminance_lag_60min"
]

all_environment_features = environment_features + illuminance_features

all_numeric_features = [
    col for col in all_environment_features
    if col != "room_type"
]

all_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), all_numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), ["room_type"])
    ]
)

all_environment_rf = Pipeline(steps=[
    ("preprocessor", all_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

X_train_all = model_df.iloc[train_idx][all_environment_features].copy()
X_test_all = model_df.iloc[test_idx][all_environment_features].copy()

all_environment_rf.fit(X_train_all, y_train)

y_pred_all = all_environment_rf.predict(X_test_all)

print("RF with CO2, temperature, humidity and illuminance")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_all):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_all)):.4f}")
print(f"R2:   {r2_score(y_test, y_pred_all):.4f}")

RF with CO2, temperature, humidity and illuminance
MAE:  0.0339
RMSE: 0.0763
R2:   0.7781


In [40]:
from sklearn.base import clone

# Prepare features and target from the unified dataset
X_cross = model_df[feature_columns].copy()
y_cross = model_df[target_column].copy()

# Separate datasets
ashrae_mask = model_df["source"] == "ASHRAE"
robod_mask = model_df["source"] == "ROBOD"

X_ashrae = X_cross.loc[ashrae_mask]
y_ashrae = y_cross.loc[ashrae_mask]

X_robod = X_cross.loc[robod_mask]
y_robod = y_cross.loc[robod_mask]

print("ASHRAE rows:", len(X_ashrae))
print("ROBOD rows:", len(X_robod))

ASHRAE rows: 37224
ROBOD rows: 51042


In [41]:
# Experiment 1: Train on ASHRAE, test on ROBOD
ashrae_to_robod = clone(rf_model)

ashrae_to_robod.fit(X_ashrae, y_ashrae)

pred_ashrae_to_robod = ashrae_to_robod.predict(X_robod)

# Persistence baseline
baseline_robod = X_robod["occupancy_ratio"]

# Experiment 2: Train on ROBOD, test on ASHRAE
robod_to_ashrae = clone(rf_model)

robod_to_ashrae.fit(X_robod, y_robod)

pred_robod_to_ashrae = robod_to_ashrae.predict(X_ashrae)

# Persistence baseline
baseline_ashrae = X_ashrae["occupancy_ratio"]


# Collect results
cross_results = []

experiments = [
    (
        "ASHRAE -> ROBOD",
        y_robod,
        pred_ashrae_to_robod,
        baseline_robod
    ),
    (
        "ROBOD -> ASHRAE",
        y_ashrae,
        pred_robod_to_ashrae,
        baseline_ashrae
    )
]

for name, actual, predicted, baseline in experiments:

    cross_results.append({
        "experiment": name,
        "rows": len(actual),
        "RF_MAE": mean_absolute_error(actual, predicted),
        "Baseline_MAE": mean_absolute_error(actual, baseline),
        "RF_RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "Baseline_RMSE": np.sqrt(mean_squared_error(actual, baseline)),
        "RF_R2": r2_score(actual, predicted)
    })

display(pd.DataFrame(cross_results).round(4))

,experiment,rows,RF_MAE,Baseline_MAE,RF_RMSE,Baseline_RMSE,RF_R2
0,ASHRAE -> ROBOD,51042,0.0432,0.0226,0.0771,0.0595,0.6389
1,ROBOD -> ASHRAE,37224,0.0630,0.0381,0.1298,0.0977,0.6184


In [42]:
distribution_comparison = model_df.groupby("source")[
    "target_occupancy_ratio_30min"
].describe()

display(distribution_comparison.round(4))

,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
ASHRAE,37224.0,0.1167,0.2101,0.0,0.0,0.0119,0.125,2.375
ROBOD,51042.0,0.0774,0.1282,0.0,0.0,0.0000,0.120,0.950


In [43]:
environment_comparison = model_df.groupby("source")[
    ["co2", "temperature", "humidity", "illuminance"]
].agg(["count", "mean", "std", "min", "median", "max"])

display(environment_comparison.round(2))

co2                                          temperature         \
        count    mean     std     min  median      max       count   mean   
source                                                                      
ASHRAE  34845  543.06  106.85  405.76  516.80  1207.68       15540  22.75   
ROBOD   50957  468.82   61.23  400.00  451.87  1489.36       51042  27.20   

                     ... humidity                      illuminance          \
         std    min  ...      std    min median    max       count    mean   
source               ...                                                     
ASHRAE  1.61  16.50  ...     9.96  22.75  41.96  75.69       18491  236.95   
ROBOD   1.29  20.44  ...     8.91  45.39  65.49  93.00       50957  137.25   

                                       
           std  min  median       max  
source                                 
ASHRAE  207.26  0.0  207.04   2064.64  
ROBOD   292.50  0.0   28.96  10698.77  

[2 rows x 24 columns]

In [44]:
environment_columns = [
    "co2",
    "temperature",
    "humidity",
    "illuminance"
]

for feature in environment_columns:

    print(f"\n{'=' * 15} {feature.upper()} {'=' * 15}")

    display(
        model_df.groupby("source")[feature]
        .agg(["count", "mean", "std", "min", "median", "max"])
        .round(2)
    )


=============== CO2 ===============


,count,mean,std,min,median,max
source,,,,,,
ASHRAE,34845,543.06,106.85,405.76,516.80,1207.68
ROBOD,50957,468.82,61.23,400.00,451.87,1489.36



=============== TEMPERATURE ===============


,count,mean,std,min,median,max
source,,,,,,
ASHRAE,15540,22.75,1.61,16.50,22.50,28.4
ROBOD,51042,27.20,1.29,20.44,27.43,34.0



=============== HUMIDITY ===============


,count,mean,std,min,median,max
source,,,,,,
ASHRAE,6126,42.96,9.96,22.75,41.96,75.69
ROBOD,50957,68.42,8.91,45.39,65.49,93.00



=============== ILLUMINANCE ===============


,count,mean,std,min,median,max
source,,,,,,
ASHRAE,18491,236.95,207.26,0.0,207.04,2064.64
ROBOD,50957,137.25,292.50,0.0,28.96,10698.77


In [45]:
sensor_columns = [
    "co2",
    "temperature",
    "humidity",
    "illuminance"
]

targets = [
    "occupancy_ratio",
    "target_occupancy_ratio_30min"
]

for source in ["ASHRAE", "ROBOD"]:

    source_df = model_df[model_df["source"] == source]

    print(f"\n========== {source} ==========")

    for sensor in sensor_columns:

        for target in targets:

            valid = source_df[[sensor, target]].dropna()

            correlation = valid[sensor].corr(valid[target])

            print(
                f"{sensor} vs {target}: "
                f"correlation = {correlation:.3f}, "
                f"rows = {len(valid)}"
            )


========== ASHRAE ==========
co2 vs occupancy_ratio: correlation = 0.579, rows = 34845
co2 vs target_occupancy_ratio_30min: correlation = 0.516, rows = 34845
temperature vs occupancy_ratio: correlation = -0.122, rows = 15540
temperature vs target_occupancy_ratio_30min: correlation = -0.141, rows = 15540
humidity vs occupancy_ratio: correlation = 0.093, rows = 6126
humidity vs target_occupancy_ratio_30min: correlation = 0.105, rows = 6126
illuminance vs occupancy_ratio: correlation = 0.018, rows = 18491
illuminance vs target_occupancy_ratio_30min: correlation = -0.001, rows = 18491

========== ROBOD ==========
co2 vs occupancy_ratio: correlation = 0.293, rows = 50957
co2 vs target_occupancy_ratio_30min: correlation = 0.313, rows = 50957
temperature vs occupancy_ratio: correlation = 0.193, rows = 51042
temperature vs target_occupancy_ratio_30min: correlation = 0.182, rows = 51042
humidity vs occupancy_ratio: correlation = 0.162, rows = 50957
humidity vs target_occupancy_ratio_30min: cor